# Data Handling Lab: NumPy, Pandas & Matplotlib

**Goal:** practice loading, cleaning, exploring, and visualizing a small dataset.

### Open in Google Colab
1. Download this notebook from your workspace.
2. Go to [Google Colab](https://colab.research.google.com/) and choose **File → Upload notebook**.
3. Run cells from top to bottom with **Runtime → Run all** (or `Shift+Enter` cell by cell).

No external files, Kaggle account, or package installation is needed. The sample data is generated below.

## Learning objectives
- Use NumPy arrays for basic numerical calculations.
- Inspect, select, filter, sort, and summarize tabular data with Pandas.
- Handle missing values and duplicate rows.
- Create clear plots with Matplotlib.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 1. Create a small practice dataset
This reproducible, fictional dataset contains 24 students. We intentionally add a missing score and a duplicate so you can practice cleaning.

In [ ]:
rng = np.random.default_rng(42)
n = 24

students = pd.DataFrame({
    "Student_ID": np.arange(1, n + 1),
    "Study_Hours": np.round(rng.uniform(1, 8, n), 1),
    "Attendance_Pct": rng.integers(60, 101, n),
    "Sleep_Hours": np.round(rng.uniform(5, 9, n), 1),
    "Group": np.resize(["A", "B", "C"], n),
})

# Scores are related to study time, with a little random variation.
students["Score"] = np.clip(
    np.round(45 + students["Study_Hours"] * 6 + rng.normal(0, 8, n)), 0, 100
).astype(float)
students.loc[students["Student_ID"] == 5, "Score"] = np.nan
students = pd.concat([students, students.iloc[[0]]], ignore_index=True)

students.head()

## 2. Inspect the data with Pandas
Try these commands when exploring a new dataset: `head`, `shape`, `info`, `dtypes`, and `describe`.

In [ ]:
print("Rows and columns:", students.shape)
print("\nColumn types:\n", students.dtypes)
print("\nDataset summary:")
students.describe(include="all")

In [ ]:
students.info()
print("\nMissing values per column:\n", students.isna().sum())
print("\nDuplicate rows:", students.duplicated().sum())

## 3. NumPy practice
NumPy arrays are useful for fast numerical operations. The array below uses study hours from the dataset.

In [ ]:
hours = students["Study_Hours"].to_numpy()
print("First five study-hour values:", hours[:5])
print("Mean study hours:", round(np.mean(hours), 2))
print("Maximum study hours:", np.max(hours))
print("Students studying at least 5 hours:", np.sum(hours >= 5))

## 4. Select, filter, and sort rows
Use `.loc` for label-based selection and Boolean conditions for filtering.

In [ ]:
# Select a few columns and show students with at least 6 study hours.
high_study = students.loc[
    students["Study_Hours"] >= 6,
    ["Student_ID", "Group", "Study_Hours", "Score"],
]
high_study.sort_values("Study_Hours", ascending=False).head(10)

## 5. Clean the dataset
Here we remove exact duplicate rows and fill missing scores with the median score. In a real study, choose a missing-data strategy based on what the data means.

In [ ]:
clean = students.drop_duplicates().copy()
score_median = clean["Score"].median()
clean["Score"] = clean["Score"].fillna(score_median)

print("Rows before cleaning:", len(students))
print("Rows after removing duplicates:", len(clean))
print("Missing values remaining:\n", clean.isna().sum())

## 6. Add a column and summarize groups
A derived column can label students as passing (score of 60 or above). `groupby` compares results across groups.

In [ ]:
clean["Passed"] = clean["Score"] >= 60
group_summary = clean.groupby("Group").agg(
    Students=("Student_ID", "count"),
    Mean_Score=("Score", "mean"),
    Mean_Study_Hours=("Study_Hours", "mean"),
    Pass_Rate=("Passed", "mean"),
).round(2)

group_summary

## 7. Visualize with Matplotlib
Each chart has a title and labeled axes. Run each cell to display a separate plot.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(clean["Score"], bins=8, color="steelblue", edgecolor="white")
plt.title("Distribution of Student Scores")
plt.xlabel("Score")
plt.ylabel("Number of students")
plt.grid(axis="y", alpha=0.25)
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
for group, data in clean.groupby("Group"):
    plt.scatter(data["Study_Hours"], data["Score"], label=f"Group {group}", alpha=0.8)
plt.title("Study Hours and Score")
plt.xlabel("Study hours")
plt.ylabel("Score")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
mean_scores = clean.groupby("Group")["Score"].mean().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(mean_scores.index, mean_scores.values, color=["#4C78A8", "#F58518", "#54A24B"])
plt.title("Average Score by Group")
plt.xlabel("Group")
plt.ylabel("Average score")
plt.ylim(0, 100)
plt.grid(axis="y", alpha=0.25)
plt.show()